# Hopnot · Kaggle GPU 加速

每周 30h 免费 GPU（Tesla P100/T4），适合大规模参数扫描。

## 准备

1. 在 Notebook 右侧点 **+Add Data**，上传你的知识文件夹（含 `knowledge_xxx.txt` + `queries_xxx.txt`）  
2. 记下数据集的路径，默认在 `/kaggle/input/你的数据集名/`  
3. 选择 GPU 加速器：Notebook 设置 → Accelerator → **GPU T4 x2**

---

In [ ]:
# Cell 1: 安装依赖（首次运行约 3 分钟）
!pip install -q modelscope sentence-transformers faiss-cpu
!apt-get install -y -qq graphviz 2>/dev/null
!echo "依赖安装完成"

In [ ]:
# Cell 2: 检查 GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Cell 3: 克隆 Hopnot
!git clone https://github.com/IjalG/Hopnot.git /kaggle/working/Hopnot
import sys
sys.path.insert(0, "/kaggle/working/Hopnot")

---
## 用法示例

修改下面的路径为你的实际数据路径。

In [ ]:
# Cell 4a: 参数扫描（GPU 加速）
# 修改 DATA_PATH 为你的数据集在 Kaggle 中的路径
DATA_PATH = "/kaggle/input/你的数据集/data01"

!cd /kaggle/working/Hopnot && python tools/sweep.py \
    --scan {DATA_PATH} \
    --embedding qwen3 \
    --device cuda \
    --output /kaggle/working/results.json

In [ ]:
# Cell 4b: 可视化导出
DATA_PATH = "/kaggle/input/你的数据集/data01"

!cd /kaggle/working/Hopnot && python tools/export.py \
    --scan {DATA_PATH} \
    --format dot \
    --outdir /kaggle/working/graphs \
    --render

In [ ]:
# Cell 5: 查看结果
import json
try:
    with open("/kaggle/working/results.json") as f:
        results = json.load(f)
    print(f"共扫描 {len(results)} 个参数组合")
    print(f"最佳召回率: {results[0]['avg_recall']:.3f}")
    print(f"参数: {', '.join(f'{k}={results[0][k]}' for k in ['recall_threshold','restart_prob','decision_threshold','num_seeds'])}")
except FileNotFoundError:
    print("还没有结果文件，先运行 Cell 4a")

---
## 下载结果

在 Kaggle 右侧栏 **Data** → **Output** 中下载 `results.json` 和 `graphs/` 目录。

## 提示

- GPU 版本 Qwen3-Embedding 约 10 秒加载，之后每次评估 ~2 秒
- 540 个参数组合在 GPU 上约 20 分钟跑完
- Kaggle 每周 30h，够跑 90 次全量扫描